# Per-Muscle and Overall Average Metrics — P* Subjects Only

Identical to `algos_avg_results.ipynb` but **only rows where the subject ID
starts with `P` (e.g. P004, P005 …) are included** in the averages.
HV* subjects are excluded.

Subject identification works for both CSV formats:
- New format: uses the `subject` column directly.
- Old format: extracts the subject from the `pred_label` filename.

In [ ]:
import pathlib
import re
import os
import numpy as np
import pandas as pd

EVAL_DIR    = pathlib.Path(r'C:\Projects\dissector\eval_notebooks')
SUMMARY_DIR = EVAL_DIR / 'summary_results_P_only'
SUMMARY_DIR.mkdir(exist_ok=True)

# ── Canonical muscle names ────────────────────────────────────────────────────
MUSCLE_ALIASES = {
    'r_gracilis':  'R_gracilis',
    'l_gracilis':  'L_gracilis',
    'r_sartorius': 'R_sartorius',
    'l_sartorius': 'L_sartorius',
    'r_sart':      'R_sartorius',
    'l_sart':      'L_sartorius',
}

# ── Canonical metric names ────────────────────────────────────────────────────
# inter_slice_dice_* MUST come before dice (they contain the substring "dice")
# dice pattern is anchored (^...$) so it never matches a longer column name
_METRIC_RE = [
    ('inter_slice_dice_pred', re.compile(r'inter_slice_dice_pred',         re.I)),
    ('inter_slice_dice_gt',   re.compile(r'inter_slice_dice_gt',           re.I)),
    ('dice',                  re.compile(r'^(?:lower_)?dice$',             re.I)),
    ('hausdorff',             re.compile(r'hausdorff',                     re.I)),
    ('jaccard',               re.compile(r'jaccard',                       re.I)),
    ('volume_similarity',     re.compile(r'volume_similarity',             re.I)),
    ('false_negative',        re.compile(r'false.?neg|falseNeg',           re.I)),
    ('false_positive',        re.compile(r'false.?pos|falsePo',            re.I)),
    ('bce',                   re.compile(r'\bbce\b|binary_cross_entropy',  re.I)),
    ('boundary_iou_3d',       re.compile(r'boundary_iou',                  re.I)),
]

_MUSCLE_PREFIX_RE      = re.compile(r'^[RrLl]_(?:gracilis|sartorius|sart)_', re.I)
_SUBJECT_FROM_FILE_RE  = re.compile(r'^(.+?)_(WATER|FATFRACTION)_stack', re.I)
_FLOAT64_MAX           = np.finfo(np.float64).max


def canonical_metric(col):
    bare = _MUSCLE_PREFIX_RE.sub('', col).rstrip(':')
    for name, pat in _METRIC_RE:
        if pat.search(bare):
            return name
    return None


def extract_muscle(stem):
    s = stem.lower()
    for alias in sorted(MUSCLE_ALIASES, key=len, reverse=True):
        if f'_{alias}_' in s or s.endswith(f'_{alias}'):
            return MUSCLE_ALIASES[alias]
    return None


def get_subjects(df):
    """Return a Series of subject IDs for each row, handling both CSV formats."""
    if 'subject' in df.columns:
        return df['subject'].astype(str)
    # read_csv(index_col=0) puts 'subject' into the index rather than columns;
    # this happens for all CSVs whose first column is 'subject' (e.g. dixon).
    if df.index.name == 'subject':
        return df.index.astype(str)
    for col in ('pred_label', 'pred_file', 'image'):
        if col in df.columns:
            def _extract(val):
                m = _SUBJECT_FROM_FILE_RE.match(os.path.basename(str(val)))
                return m.group(1) if m else None
            return df[col].apply(_extract).astype(str)
    return pd.Series([''] * len(df), index=df.index)


def process_algorithm(label, results_dir):
    """Return a summary DataFrame for P* subjects only, or None if no CSVs found."""
    results_dir = pathlib.Path(results_dir)
    csv_files   = sorted(results_dir.glob('*.csv'))
    if not csv_files:
        print(f'  [skip] no CSVs in {results_dir}')
        return None

    rows = []
    for csv_path in csv_files:
        muscle = extract_muscle(csv_path.stem)
        if muscle is None:
            print(f'  [skip] could not identify muscle in {csv_path.name}')
            continue

        df = pd.read_csv(csv_path, index_col=0)

        # ── Filter to P* subjects only ────────────────────────────────────
        subjects = get_subjects(df)
        mask     = subjects.str.startswith('P', na=False)
        df       = df[mask]

        if df.empty:
            print(f'  [skip] no P* rows in {csv_path.name}')
            continue

        print(f'  {csv_path.name}: {mask.sum()} P* rows (of {len(mask)} total)')

        metric_vals = {}
        for col in df.columns:
            metric_name = canonical_metric(col)
            if metric_name is None:
                continue

            # ── SAFE NUMERIC CONVERSION ──────────────────────────────────
            s = pd.to_numeric(df[col], errors='coerce').to_numpy(dtype=np.float64)

            # Replace inf/-inf AND float64-max sentinel values (stored overflow)
            # with NaN so nanmean never overflows.
            s = np.where(np.isfinite(s) & (np.abs(s) < _FLOAT64_MAX), s, np.nan)

            finite = np.isfinite(s)
            if not finite.any():
                print(f'  [ALL NON-FINITE] {csv_path.name} | {col}')
                continue

            vals    = s[finite]
            min_val = np.min(vals)
            max_val = np.max(vals)

            if metric_name in {'dice', 'jaccard', 'volume_similarity'}:
                if max_val > 1.0 or min_val < -0.1:
                    print(
                        f'  [SCALE ERROR] {csv_path.name} | {col} '
                        f'| expected ~[0,1], got min={min_val:.3f}, max={max_val:.3f}'
                    )

            if metric_name in {'false_negative', 'false_positive'}:
                if max_val > 1e3:
                    print(
                        f'  [COUNT-LIKE FN/FP] {csv_path.name} | {col} '
                        f'| max={max_val:.3e} (likely unnormalized)'
                    )

            metric_vals[metric_name] = np.nanmean(s, dtype=np.float64)

        # ── DERIVED: inter-slice dice ratio (pred / gt) ──────────────────
        # Normalises prediction smoothness against the object's inherent
        # slice-to-slice variation; 1.0 = as consistent as the GT shape.
        pred = metric_vals.get('inter_slice_dice_pred')
        gt   = metric_vals.get('inter_slice_dice_gt')
        if pred is not None and gt is not None and gt > 0:
            metric_vals['inter_slice_dice_ratio'] = pred / gt

        row = {'muscle': muscle}
        row.update(metric_vals)
        rows.append(row)

    if not rows:
        return None

    summary = pd.DataFrame(rows).set_index('muscle')
    summary.insert(0, 'algorithm', label)

    numeric = summary.select_dtypes(include='number').astype(np.float64)
    overall = numeric.mean().rename('Overall_Mean')
    overall['algorithm'] = label
    summary = pd.concat([summary, overall.to_frame().T])
    summary.index.name = 'muscle'
    return summary


print('Helpers ready.')

In [ ]:
# ── Algorithm registry ────────────────────────────────────────────────────────
REGISTRY = [
    ('MuscleMap Thigh (water)',           'muscle_map_thigh/results_water'),
    ('MuscleMap Thigh (fat fraction)',    'muscle_map_thigh/results_fat_frac'),
    ('MuscleMap WB (water)',              'muscle_map_wb/results_water'),
    ('MuscleMap WB (fat fraction)',       'muscle_map_wb/results_fat_frac'),
    ('MM WB + MedSAM bbox (water)',       'muscle_map_wb_boxes_medsam/results_water'),
    ('MM WB + MedSAM bbox (fat fraction)','muscle_map_wb_boxes_medsam/results_fatfrac'),
    ('MM WB + MedSAM mask (water)',        'muscle_map_wb_masks_medsam/results_water'),
    ('MM WB + MedSAM mask (fat fraction)', 'muscle_map_wb_masks_medsam/results_fat_frac'),
    ('MM WB + SLM-SAM2 (water)',          'muscle_map_wb+slmsam/results_water'),
    ('MM WB + SLM-SAM2 (fat fraction)',   'muscle_map_wb+slmsam/results_fat_frac'),
    ('Dafne (water)',                     'dafne/results_water'),
    ('Dafne (fat fraction)',              'dafne/results_fat_frac'),
    ('Dafne + MedSAM (water)',            'dafne_and_medsam/results_water'),
    ('Dafne + MedSAM (fat fraction)',     'dafne_and_medsam/results_fat_frac'),
    ('Hirriririir (water)',               'multimodal-multiethnic/results_water'),
    ('Hirriririir (fat fraction)',        'multimodal-multiethnic/results_fat_frac'),
    ('MuSeg (water)',                     'museg/results_water'),
    ('MuSeg (fat fraction)',              'museg/results_fat_frac'),
    ('MuSeg (dixon)',                     'museg/results_dixon'),
    ('MedCLIP-SAMv2 (water)',             'medclipsamv2/results_water'),
    ('MedCLIP-SAMv2 (fat fraction)',      'medclipsamv2/results_fat_frac'),
    ('MedCLIP-SAMv2 + MM Boxes (water)',        'medclipsamv2plusboxes/results_water'),
    ('MedCLIP-SAMv2 + MM Boxes (fat fraction)', 'medclipsamv2plusboxes/results_fat_frac'),
    ('MedCLIP-SAMv2 Text+Boxes (water)',        'medclipsamv2textboxes/results_water'),
    ('MedCLIP-SAMv2 Text+Boxes (fat fraction)', 'medclipsamv2textboxes/results_fat_frac'),
    ('MedSegDiff (both channels)',                'medsegdiff/results_channels'),
    ('MedSegDiff (fat fraction)',         'medsegdiff/results_fat_frac'),
    ('MedSegDiff (water)',                'medsegdiff/results_water'),
]

print(f'{len(REGISTRY)} entries in registry.')
for label, rdir in REGISTRY:
    path = EVAL_DIR / rdir
    n = len(list(path.glob('*.csv'))) if path.exists() else 0
    status = f'{n} CSVs' if path.exists() else 'DIR MISSING'
    print(f'  {label}: {status}')

In [ ]:
# ── Process all algorithms (P* subjects only) ─────────────────────────────────
summaries = {}

for label, rdir in REGISTRY:
    print(f'\n── {label} ──')
    df = process_algorithm(label, EVAL_DIR / rdir)
    if df is None:
        continue
    summaries[label] = df

    num_cols = df.select_dtypes(include='number').columns
    display(df.reset_index().style.format('{:.4f}', subset=num_cols).hide(axis='index'))

    safe_name = re.sub(r'[^\w]+', '_', label).strip('_').lower()
    out_path  = SUMMARY_DIR / f'{safe_name}_avg_metrics_P.csv'
    df.to_csv(out_path, float_format='%.4f')
    print(f'  Saved -> {out_path}')

print(f'\nProcessed {len(summaries)}/{len(REGISTRY)} algorithms.')

In [ ]:
# ── Combined Overall Means (P* only) ─────────────────────────────────────────
overall_rows = [
    df.loc[['Overall_Mean']]
    for df in summaries.values()
    if 'Overall_Mean' in df.index
]

combined = pd.concat(overall_rows)
combined.index = [row['algorithm'] for _, row in combined.iterrows()]
combined.index.name = 'algorithm'
combined = combined.drop(columns='algorithm')
combined = combined.drop(columns=['inter_slice_dice_pred', 'inter_slice_dice_gt'], errors='ignore')

num_cols = combined.select_dtypes(include='number').columns
display(
    combined.reset_index()
    .style
    .format('{:.4f}', subset=num_cols)
    .hide(axis='index')
    .background_gradient(subset=['dice'], cmap='RdYlGn', axis=0)
    .background_gradient(subset=['hausdorff'], cmap='RdYlGn_r', axis=0)
)

out_combined = SUMMARY_DIR / 'overall_means_P_only.csv'
combined.to_csv(out_combined, float_format='%.4f')
print('Saved ->', out_combined)